# 🎙️ Advanced Speaker Diarization Suite (Pyannote 4.x)

This notebook provides a production-grade environment for performing **Speaker Diarization** (who spoke when) on multi-speaker audio recordings. It is fully optimized for **Google Colab (T4 GPU runtime)** and resolves common library backend conflicts (such as Torchaudio crashes in Python 3.12) and sample rate mismatch errors.

---

### ✨ Features
1. **Pyannote 4.x Engine**: Leverages the latest `pyannote/speaker-diarization-community-1` pipeline with optimized GPU execution.
2. **Robust Audio Ingestion**: Automated FFmpeg conversion of incoming formats (`.m4a`, `.mp3`, `.wav`, etc.) to a standardized **16kHz mono PCM WAV**, preventing Pyannote's `ValueError` sample count mismatch crashes.
3. **Dynamic Output Naming**: Automatically extracts the filename from the input path and names all exported files accordingly, preventing different audios from overwriting each other.
4. **Smart Timeline Post-Processing**: Groups adjacent speaker turns and merges small gaps (e.g. pauses) to build a readable, clean timeline.
5. **Speaker Statistics**: Generates metrics including total speech time, speech duration percentage, and turn counts per speaker.
6. **Multi-Format Export**: Generates premium Markdown reports, CSV files (for spreadsheets), JSON files (for developers), and standard RTTM files.

## ⚙️ Step 0: Setup & Core Dependencies

Make sure your Google Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

Run the cell below to install the necessary packages. It uses version pins and settings designed to avoid conflict errors in Python 3.12.

In [ ]:
# 1. Uninstall pre-installed incompatible torchao to prevent import conflicts with PyTorch
!pip uninstall -y -q torchao

# 2. Install Pyannote 4.x and audio libraries using pre-compiled binary wheels
# We use numpy>=2.3 and pyannote.audio>=4.0.1 to avoid torchaudio.set_audio_backend issues in Colab's Python 3.12 environment
!pip install -q --upgrade "numpy>=2.3" "pyannote.audio>=4.0.1" soundfile pandas --prefer-binary

print("\n[IMPORTANT] If prompted by Google Colab, click the 'RESTART SESSION' or 'RESTART RUNTIME' button to apply package updates.")

## 🔑 Step 1: Configuration & Hugging Face Access

Before running the cells below, configure your Hugging Face token:
1. Go to your Hugging Face settings and create a **Read Access Token**.
2. Visit **[pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)** and accept the user conditions.
3. In Google Colab, open the Secrets manager (the key icon 🔑 in the left sidebar).
4. Add a secret named **`HF_TOKEN`** and paste your token.
5. Toggle the **"Notebook access"** switch to **ON** for this secret!

In [ ]:
import torch
from google.colab import userdata

# Retrieve HF Token
try:
    hf_token = userdata.get('HF_TOKEN')
    print("[SUCCESS] Retrieved Hugging Face access token (HF_TOKEN) from Secrets.")
except Exception:
    hf_token = None
    print("[WARNING] 'HF_TOKEN' not found in Secrets. If Pyannote loading fails, please configure it.")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")
if device.type == "cpu":
    print("[WARNING] Running on CPU. This will be SIGNIFICANTLY slower. Please switch your runtime to T4 GPU (Runtime -> Change runtime type).")

## 🎵 Step 2: Audio Ingestion & Standardization

Specify the path to your raw audio file. 

### 💡 Input Path Reference:
- **Google Drive File**: Mount Drive and enter a path starting with `/content/drive/MyDrive/` (e.g. `/content/drive/MyDrive/MyAudioFiles/interview.m4a`).
- **Local Upload**: Upload a file directly to the Colab sidebar using the folder icon and enter a path starting with `/content/` (e.g. `/content/recording.wav`).

The notebook automatically extracts the name of your file to use it dynamically in all outputs, preventing new recordings from overwriting previous ones. It also runs FFmpeg to standardize the audio into a **16kHz mono WAV** file, bypassing Pyannote's sample rate crash issues.

In [ ]:
from google.colab import drive
import os
import subprocess

# Mount Google Drive (Required if loading files from your Drive folders)
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception as e:
    print("[INFO] Drive mount skipped or already mounted.")

# @markdown ### Audio File Selection
# @markdown Enter the absolute path to your audio file. Examples:
# @markdown - Drive file: `/content/drive/MyDrive/Sample Audio Files/MarauliKhurad2.m4a`
# @markdown - Direct upload: `/content/my_interview.wav`
audio_file_path = "/content/drive/MyDrive/Sample Audio Files/MarauliKhurad2.m4a" # @param {type:"string"}

if not os.path.exists(audio_file_path):
    print(f"\n[ERROR] File not found at: '{audio_file_path}'")
    print("Please check the path inside your Google Drive or local file explorer and run this cell again.")
else:
    # Extract filename information for dynamic naming
    audio_filename = os.path.basename(audio_file_path)
    audio_name_only, _ = os.path.splitext(audio_filename)
    
    # Define dynamic path for processed audio
    processed_audio_path = f"./{audio_name_only}_processed_16k.wav"

    print(f"\n[SUCCESS] Audio file verified: '{audio_file_path}'")
    print("Standardizing audio to 16kHz mono WAV format via FFmpeg...")
    
    try:
        # Convert audio to mono 16kHz WAV format (PCM 16-bit)
        cmd = [
            "ffmpeg", "-y", "-i", audio_file_path,
            "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le",
            processed_audio_path
        ]
        # Run command and hide normal output, show only on error
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print(f"[SUCCESS] Pre-processing complete. Saved standard WAV to: {processed_audio_path}")
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] FFmpeg preprocessing failed!")
        print(f"stderr: {e.stderr.decode()}")
        print("Falling back to raw audio file (Note: This may cause Pyannote crop ValueErrors).")
        processed_audio_path = audio_file_path

## 👥 Step 3: Load & Run Speaker Diarization

Load Pyannote's `community-1` model and execute the pipeline on your processed audio file.

By default, `num_speakers` is set to `0`, which lets Pyannote's clustering engine automatically detect the number of speakers from the audio. If you know the exact number of speakers in advance, you can specify it to improve clustering accuracy.

In [ ]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

# @markdown ### Speaker Clustering Constraints (Optional)
# @markdown Set to 0 to let Pyannote estimate the number of speakers automatically (highly recommended for generic files).
num_speakers = 0 # @param {type:"integer"}
min_speakers = 0 # @param {type:"integer"}
max_speakers = 0 # @param {type:"integer"}

print("Initializing Pyannote Speaker Diarization Pipeline...")
try:
    diarization_pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-community-1",
        token=hf_token
    )
    diarization_pipeline.to(device)
    print("[SUCCESS] Diarization pipeline loaded successfully.")
except Exception as e:
    print(f"[ERROR] Failed to load Pyannote pipeline. Make sure you accepted user conditions on Hugging Face and your HF_TOKEN is valid.")
    raise e

print("\n--- Running Speaker Diarization ---")

# Setup pipeline params
diarization_params = {}
if num_speakers > 0:
    diarization_params["num_speakers"] = num_speakers
else:
    if min_speakers > 0:
        diarization_params["min_speakers"] = min_speakers
    if max_speakers > 0:
        diarization_params["max_speakers"] = max_speakers

with ProgressHook() as hook:
    diarization_output = diarization_pipeline(processed_audio_path, hook=hook, **diarization_params)

# Extract speaker segments
raw_speaker_segments = []
for turn, speaker in diarization_output.speaker_diarization:
    raw_speaker_segments.append({
        "start": turn.start,
        "end": turn.end,
        "speaker": speaker
    })

print(f"\nDiarization complete. Identified {len(set(s['speaker'] for s in raw_speaker_segments))} unique speaker(s).")
print(f"Total raw speaker turns: {len(raw_speaker_segments)}")

## 📊 Step 4: Timeline Processing & Speaker Analytics

This step refines the raw timelines by:
1. **Merging small gaps**: Merges consecutive turns of the same speaker if the gap between them is less than a certain threshold (e.g. 1.5 seconds) to create cleaner timelines.
2. **Generating Analytics**: Calculates the total talking duration, turn count, and speech percentage for each speaker.

In [ ]:
# @markdown ### Timeline Processing Config
# @markdown Merge adjacent turns of the same speaker if the silence gap is less than this threshold (seconds):
max_merge_gap = 1.5 # @param {type:"number"}

# Merges consecutive speaker segments of the same speaker if the gap between them is small
def merge_speaker_segments(segments, max_gap=1.5):
    if not segments:
        return []
    sorted_segs = sorted(segments, key=lambda x: x["start"])
    merged = []
    current = sorted_segs[0].copy()
    for next_seg in sorted_segs[1:]:
        if next_seg["speaker"] == current["speaker"] and (next_seg["start"] - current["end"]) <= max_gap:
            current["end"] = max(current["end"], next_seg["end"])
        else:
            merged.append(current)
            current = next_seg.copy()
    merged.append(current)
    return merged

# Process segments
speaker_segments = merge_speaker_segments(raw_speaker_segments, max_gap=max_merge_gap)

# Calculate speaker stats
speaker_stats = {}
total_speech_time = 0.0

for seg in speaker_segments:
    duration = seg["end"] - seg["start"]
    speaker = seg["speaker"]
    total_speech_time += duration
    
    if speaker not in speaker_stats:
        speaker_stats[speaker] = {
            "total_duration": 0.0,
            "turn_count": 0
        }
    speaker_stats[speaker]["total_duration"] += duration
    speaker_stats[speaker]["turn_count"] += 1

print(f"[SUCCESS] Processed speaker turns merged from {len(raw_speaker_segments)} down to {len(speaker_segments)} clean turns.")
print("\n--- Speaker Statistics Summary ---")
for spk, stats in speaker_stats.items():
    percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
    print(f"{spk}:")
    print(f"  - Total Speech Time: {stats['total_duration']:.2f}s ({percentage:.1f}%)")
    print(f"  - Turn Count: {stats['turn_count']}")

## 💾 Step 5: Export Reports & Timelines

Format the timelines and statistics into a premium Markdown report (complete with timing badges and a text-based participation chart), and export standard CSV, JSON, and RTTM formats. The filenames will dynamically include the name of the source audio file.

In [ ]:
import csv
import json
import pandas as pd
from IPython.display import Markdown, display

# Time formatting helper: convert seconds to [MM:SS.d] or [HH:MM:SS.d]
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 10)
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:01d}"
    else:
        return f"{minutes:02d}:{secs:02d}.{millis:01d}"

# Define dynamic filenames based on processed file name
rttm_path = f"{audio_name_only}_timeline.rttm"
json_output_path = f"{audio_name_only}_timeline.json"
csv_output_path = f"{audio_name_only}_timeline.csv"
md_output_path = f"{audio_name_only}_diarization_report.md"

# 1. Export standard RTTM file
with open(rttm_path, "w") as rttm_file:
    diarization_output.speaker_diarization.write_rttm(rttm_file)

# 2. Save JSON Timeline
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(speaker_segments, f, indent=4, ensure_ascii=False)

# 3. Save CSV Timeline
with open(csv_output_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Speaker", "Start_Time", "End_Time", "Timestamp_Formatted"])
    for entry in speaker_segments:
        writer.writerow([
            entry["speaker"],
            entry["start"],
            entry["end"],
            f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
        ])

# 4. Generate Premium Markdown Report
with open(md_output_path, "w", encoding="utf-8") as f:
    f.write(f"# 🎙️ Speaker Diarization Analysis Report: {audio_filename}\n\n")
    f.write("Generated using the Pyannote 4.x Speaker Diarization Pipeline.\n\n")
    
    # Add stats section
    f.write("## 📊 Speaker Participation Summary\n\n")
    f.write("| Speaker | Total Speaking Duration | Turn Count | Speech % |\n")
    f.write("| :--- | :--- | :--- | :--- |\n")
    for spk, stats in sorted(speaker_stats.items()):
        percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
        f.write(f"| **{spk}** | {stats['total_duration']:.2f}s | {stats['turn_count']} | {percentage:.1f}% |\n")
    
    # Add visual chart
    f.write("\n### 📈 Visual Share of Speech\n\n")
    for spk, stats in sorted(speaker_stats.items()):
        percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
        bars = "█" * int(percentage // 4)
        f.write(f"- **{spk}**: `{bars}` ({percentage:.1f}%)\n")
        
    f.write("\n---\n\n")
    f.write("## 📅 Cleaned Timeline\n\n")
    for entry in speaker_segments:
        time_str = f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
        f.write(f"> **{entry['speaker']}** `{time_str}`\n\n")
    f.write("---\n")

print(f"[SUCCESS] Exported reports and timelines (dynamically named):")
print(f" - Standard RTTM: {rttm_path}")
print(f" - JSON Timeline: {json_output_path}")
print(f" - CSV Timeline: {csv_output_path}")
print(f" - Markdown Report: {md_output_path}")

print("\n--- Diarization Report Preview ---")
with open(md_output_path, "r") as f:
    display(Markdown(f.read()))